# 📘 부록: 이미지 생성 모델을 4비트로 불러와 LoRA 학습하기(QLoRA)

지난 시간 부록은 이미지 생성 모델 `black-forest-labs/FLUX.2-klein-base-4B`의 트랜스포머를 bf16으로 올려 초롱이 LoRA 어댑터를 학습했습니다. 이 부록은 같은 학습 그림과 캡션, 같은 학습 방법에서 트랜스포머만 4비트(NF4)로 불러와 학습합니다(QLoRA). 지난 시간 실행 결과를 비교 기준으로 두고, 4비트에서 학습한 어댑터를 bf16 트랜스포머에 얹어 그리는 것까지 합니다.

**실행 환경:** 교안과 같은 RunPod 파드(GPU 메모리 24GB급)에서 실행합니다. 모델 파일은 약 15GB이고, 처음 불러올 때 내려받습니다.

**학습 목표**

1. **이미지 생성 모델의 트랜스포머를 `BitsAndBytesConfig`로 4비트로 불러와, 학습 메모리와 갱신당 시간을 같은 GPU에서 bf16 트랜스포머와 비교합니다.**
2. **같은 시드로 bf16 Base와 4비트 Base의 학습 전 그림을 그려, 양자화가 그림에 주는 영향을 확인합니다.**
3. **4비트 트랜스포머에 지난 시간과 같은 방법으로 초롱이 어댑터를 학습하고, 손실과 고른 체크포인트를 지난 시간 실행 결과와 비교합니다.**
4. **4비트에서 학습한 어댑터를 bf16 트랜스포머에 얹어 그립니다.**

## 1. 지난 시간 부록과 달라지는 것

이미지 생성 모델의 세 부분(텍스트 인코더, 트랜스포머, VAE), 플로 매칭 손실, 학습 방법은 지난 시간 부록과 같습니다. 교안 01이 언어 모델의 선형 층을 4비트로 불러왔듯이, 이번에는 가장 큰 트랜스포머의 선형 층만 4비트로 불러오고 그 위에 같은 LoRA 어댑터를 학습합니다.

<img src="images/30_diffusion_qlora.png" width="1050" alt="트랜스포머만 4비트로 불러와 LoRA 어댑터를 학습합니다. 텍스트 인코더는 프롬프트 벡터를 만든 뒤 GPU에서 빼고, 학습 중 GPU에는 4비트 트랜스포머, VAE, 어댑터, 활성값이 남습니다.">

| 부분 | 지난 시간 부록 | 이번 부록 |
|---|---|---|
| 텍스트 인코더(Qwen3-4B) | bf16, 프롬프트 벡터를 만든 뒤 뺌 | 같음 |
| 트랜스포머의 선형 층 | bf16 | 4비트 NF4(이중 양자화, 계산은 bf16) |
| 트랜스포머의 RMSNorm 가중치(query, key 정규화) | bf16 | fp32(`prepare_model_for_kbit_training`) |
| VAE | bf16 | 같음 |
| LoRA 어댑터 | float32, r 16, lora_alpha 16, 같은 대상 층 | 같음 |
| 학습 | 학습률 1e-4, 1000번 갱신, 갱신마다 그림 한 장, 체크포인트 4개 | 같음 |
| 그림 생성 | bf16 트랜스포머 | 학습 중에는 4비트 트랜스포머, 학습 뒤에는 bf16 트랜스포머에 어댑터를 얹어 그림 |

- **DreamBooth, 캡션, 트리거 단어 `Chorong`:** 지난 시간 부록 1절과 같습니다.
- **어댑터 세기와 prior preservation:** 지난 시간 부록 7·9절과 같고, 이 부록은 세기 1.0으로만 그립니다.
- **비교 기준:** 지난 시간 부록을 RunPod NVIDIA RTX PRO 4000 Blackwell(GPU 메모리 23.4GB)에서 실행한 결과입니다. 이번 GPU는 다를 수 있으므로, 메모리와 시간은 같은 GPU에서 bf16 트랜스포머와 4비트 트랜스포머를 직접 측정해 비교합니다.

## 2. 실습 준비

### 2-1. 라이브러리와 경로

지난 시간 부록과 같은 준비 셀입니다. 4비트 양자화에는 교안 01과 같은 `bitsandbytes`가 필요합니다.

In [ ]:
import json
import math
import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

material_dir = Path(".")  # data가 있는 단원 폴더
data_dir = material_dir / "data" / "부록_이미지"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: 이 부록은 GPU(cuda)가 있어야 합니다. CPU로는 갱신 한 번에 몇 분이 걸립니다
device = torch.device("cuda")
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
print("계산 장치:", device, "|", torch.cuda.get_device_name(0), "| GPU 메모리(GB):", round(total_gb, 1))

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다
pd.set_option("display.max_colwidth", None)  # 긴 글자가 잘리지 않게 합니다

### 2-2. 학습 그림과 캡션

지난 시간 부록과 같은 초롱이 그림 42장과 캡션(`captions.json`)입니다.

<img src="data/부록_이미지/chorong_reference.png" width="1050" alt="초롱이 설정화: 같은 캐릭터를 정면, 3/4 측면, 옆면, 뒷모습으로 그렸습니다. 귀는 접부채 모양이고 꼬리 끝에 초롱이 달려 있습니다.">

In [ ]:
# 그림 목록, 출처 정보(작품, 라이선스, 저작자 표시), 그림마다 붙인 캡션을 읽습니다
image_dir = data_dir / "chorong"
image_paths = sorted(image_dir.glob("*.jpg"))
source_info = json.loads((data_dir / "sources.json").read_text(encoding="utf-8"))
captions = json.loads((data_dir / "captions.json").read_text(encoding="utf-8"))  # 파일 이름 -> 캡션

print("캐릭터:", source_info["title"], "|", source_info["attribution"])
print("학습 그림 수:", len(image_paths), "| 캡션 수:", len(captions), "| 크기:", Image.open(image_paths[0]).size)

그림을 격자로 그리는 두 함수도 지난 시간 부록과 같습니다.

In [ ]:
def show_images(images, titles, n_cols, title):
    # 그림 여러 장을 n_cols칸씩 격자로 그립니다. titles는 그림마다 붙일 짧은 이름입니다
    n_rows = math.ceil(len(images) / n_cols)
    plt.figure(figsize=(n_cols * 2.8, n_rows * 2.9 + 0.5))
    for i in range(len(images)):
        plt.subplot(n_rows, n_cols, i + 1)
        plt.imshow(images[i])
        plt.title(
            titles[i],    # 칸 위에 붙일 짧은 이름
            fontsize=11,  # 칸이 작아 제목 글자도 작게 둡니다
        )
        plt.axis("off")
    plt.suptitle(
        title,        # 격자 전체의 제목
        fontsize=14,  # 칸 제목보다 크게 둡니다
    )
    plt.tight_layout()
    plt.show()


def compare_rows(rows, prompt_names, title):
    # rows: [(줄 이름, {프롬프트 이름: 그림}), ...]. 한 줄이 한 모델, 한 열이 한 프롬프트입니다
    grid_images = []
    grid_titles = []
    for row_name, images in rows:
        for name in prompt_names:
            grid_images.append(images[name])
            grid_titles.append(f"{row_name}: {name}")
    show_images(grid_images, grid_titles, len(prompt_names), title)

### 2-3. 모델 불러오기

지난 시간 부록과 같이 세 부분을 bf16으로 불러옵니다. 이 bf16 트랜스포머로 학습 전 그림(bf16 Base)을 그리고 학습 메모리를 측정한 뒤, 4비트 트랜스포머로 바꿉니다.

In [ ]:
from diffusers import Flux2KleinPipeline

# 처음 실행하면 허브에서 약 15GB를 내려받습니다(RunPod에서 몇 분)
# HF_TOKEN 이 없다는 Warning 은 로그인 없이 내려받는다는 안내이며 오류가 아닙니다
model_name = "black-forest-labs/FLUX.2-klein-base-4B"
pipe = Flux2KleinPipeline.from_pretrained(
    model_name,                  # 허브의 모델 저장소 이름
    torch_dtype=torch.bfloat16,  # 세 부분(텍스트 인코더, 트랜스포머, VAE)을 모두 bf16으로 불러옵니다
)
pipe.to(device)
pipe.set_progress_bar_config(disable=True)  # 그림마다 찍히는 진행 막대를 끕니다
print("GPU에 올라간 메모리(GB):", round(torch.cuda.memory_allocated() / 1024 ** 3, 1))

### 2-4. 프롬프트를 벡터로 바꾸고 텍스트 인코더 빼기

지난 시간 부록과 같은 확인 프롬프트 아홉(캐릭터 확인 여섯, 일반화 확인 셋)과 캡션 42개를 한 번씩만 벡터로 바꿔 두고 텍스트 인코더를 뺍니다. 프롬프트와 시드가 같아야 지난 시간 부록의 그림과 나란히 비교할 수 있습니다.

In [ ]:
character_prompts = {
    "우주 유영": "a 3D animated cartoon of Chorong the creature swimming through outer space among stars and planets",
    "수박 먹기": "a 3D animated cartoon of Chorong the creature eating a big slice of watermelon on a rooftop in summer",
    "바닷속": "a 3D animated cartoon of Chorong the creature swimming underwater in a coral reef with colorful fish",
    "열기구": "a 3D animated cartoon of Chorong the creature riding in a hot air balloon high above the clouds",
    "수채화": "a watercolor painting of Chorong the creature",
    "연필 스케치": "a pencil sketch of Chorong the creature",
}
general_prompts = {
    "부류(작은 생물)": "a 3D animated cartoon of a small creature",
    "다른 동물(개)": "a 3D animated cartoon of a dog",
    "사진(사람)": "a photo of a person reading a book in a cafe",
}
# 두 묶음을 합친 확인 프롬프트 9개
test_prompts = dict(character_prompts)
test_prompts.update(general_prompts)

# 프롬프트마다 (벡터, 토큰 위치 번호)를 한 번 계산해 둡니다
prompt_bank = {}
with torch.no_grad():
    for text in [""] + list(test_prompts.values()) + list(captions.values()):
        prompt_bank[text] = pipe.encode_prompt(
            text,           # 벡터로 바꿀 프롬프트
            device=device,  # 벡터를 만들 장치(GPU)
        )
print("프롬프트 수:", len(prompt_bank))

# 텍스트 인코더를 파이프라인에서 빼고 GPU 메모리를 비웁니다
import gc

pipe.text_encoder = None
gc.collect()
torch.cuda.empty_cache()
print("GPU에 올라간 메모리(GB):", round(torch.cuda.memory_allocated() / 1024 ** 3, 1))

### 2-5. 그림 생성 함수

지난 시간 부록과 같은 함수에 `torch.autocast` 한 줄을 더했습니다. 같은 프롬프트와 같은 시드로 그려야 학습이 바꾼 것만 비교할 수 있습니다.

In [ ]:
resolution = 512  # 학습과 생성 모두 512 x 512


def generate(prompt, seed):
    # 미리 계산한 프롬프트 벡터로 그림 한 장을 만듭니다. 같은 seed면 같은 시작 잡음입니다
    prompt_embeds, _ = prompt_bank[prompt]
    negative_embeds, _ = prompt_bank[""]
    generator = torch.Generator(device).manual_seed(seed)

    # 4비트 트랜스포머는 학습 준비 뒤 RMSNorm 가중치가 float32라 잠재 표현이 float32로 들어오므로, 공식 예제처럼 bf16 autocast 아래에서 그립니다
    with torch.autocast(
        "cuda",                # GPU 계산에 적용합니다
        dtype=torch.bfloat16,  # 트랜스포머 밖의 계산(VAE 등)도 bf16으로 맞춥니다
    ):
        result = pipe(
            prompt_embeds=prompt_embeds,             # 미리 계산한 프롬프트 벡터
            negative_prompt_embeds=negative_embeds,  # 빈 프롬프트 벡터(CFG의 비교 기준)
            height=resolution,                       # 그림 높이(학습 해상도와 같게)
            width=resolution,                        # 그림 너비(학습 해상도와 같게)
            num_inference_steps=30,                  # 잡음을 걷어 내는 횟수
            guidance_scale=4.0,                      # CFG 배율(이 모델의 기본값)
            generator=generator,                     # seed로 고정한 시작 잡음
        )
    return result.images[0]

## 3. 학습 전 그림: bf16 Base

지난 시간 부록의 학습 전 그림과 같은 조건(bf16 Base, 같은 프롬프트 아홉, 시드 0)으로 그립니다. 4비트 Base 그림과 어댑터를 얹은 그림을 이 줄과 비교합니다.

In [ ]:
start_time = time.time()
bf16_before_images = {}
for name, prompt in test_prompts.items():
    bf16_before_images[name] = generate(prompt, seed=0)
print(f"그림 {len(bf16_before_images)}장: {time.time() - start_time:.0f}초")

In [ ]:
compare_rows([("bf16 Base", bf16_before_images)], list(character_prompts), "캐릭터 확인 프롬프트, bf16 Base, 시드 0")

In [ ]:
compare_rows([("bf16 Base", bf16_before_images)], list(general_prompts), "일반화 확인 프롬프트, bf16 Base, 시드 0")

## 4. 학습 계산 준비

### 4-1. 그림을 잠재 표현으로

지난 시간 부록과 같은 함수입니다. 학습 그림은 학습 내내 바뀌지 않으므로 VAE로 잠재 표현을 한 번만 만들어 둡니다.

In [ ]:
vae = pipe.vae
# 표준화에 쓰는 평균과 표준편차(VAE의 BatchNorm 층에 저장된 값)
latents_mean = vae.bn.running_mean.view(1, -1, 1, 1).to(device)
latents_std = torch.sqrt(vae.bn.running_var.view(1, -1, 1, 1) + vae.config.batch_norm_eps).to(device)


def images_to_latents(images):
    # PIL 그림 목록 -> (그림 수, 128, 32, 32) 잠재 표현
    latents_list = []
    with torch.no_grad():
        for image in images:
            pixels = np.array(image.convert("RGB").resize((resolution, resolution), Image.LANCZOS))  # (512, 512, 3), 0~255
            pixels = torch.from_numpy(pixels).permute(2, 0, 1).float() / 127.5 - 1  # (3, 512, 512), -1~1
            latents = vae.encode(pixels.unsqueeze(0).to(device, torch.bfloat16)).latent_dist.mode()  # (1, 32, 64, 64)
            latents = Flux2KleinPipeline._patchify_latents(latents)  # (1, 128, 32, 32)
            latents_list.append((latents - latents_mean) / latents_std)
    return torch.cat(latents_list).to(torch.bfloat16)


train_latents = images_to_latents([Image.open(path) for path in image_paths])
print("학습 잠재 표현:", tuple(train_latents.shape), train_latents.dtype)

### 4-2. 손실 함수

지난 시간 부록과 같은 플로 매칭 손실 함수입니다. 함수 안의 `transformer`는 그때 이 이름에 든 트랜스포머(bf16 또는 4비트)이고, 4비트 층은 계산할 때마다 가중치를 bf16으로 복원해 곱하므로 함수는 바꾸지 않습니다.

In [ ]:
def flow_matching_loss(latents, prompt_embeds, text_ids):
    # 플로 매칭 손실(MSE)을 계산합니다
    noise = torch.randn_like(latents)
    sigmas = torch.rand(
        len(latents),   # 그림마다 잡음 비율(0~1)을 하나씩 뽑습니다
        device=device,  # 잠재 표현과 같은 GPU에 만듭니다
    )
    s = sigmas.view(-1, 1, 1, 1).to(latents.dtype)
    noisy = (1 - s) * latents + s * noise  # x_sigma = (1 - sigma) * x0 + sigma * 잡음
    target = noise - latents  # 맞혀야 할 방향 = 잡음 - x0

    latent_ids = Flux2KleinPipeline._prepare_latent_ids(latents).to(device)  # 그림 토큰 1,024개의 (행, 열) 위치 번호
    prediction = transformer(
        hidden_states=Flux2KleinPipeline._pack_latents(noisy),  # 잡음 섞인 그림 토큰 (그림 수, 1024, 128)
        timestep=sigmas,                                        # 잡음 비율(이 모델은 0~1 값을 그대로 받습니다)
        guidance=None,                                          # base 모델은 guidance 입력이 없습니다
        encoder_hidden_states=prompt_embeds,                    # 캡션의 프롬프트 벡터
        txt_ids=text_ids,                                       # 프롬프트 토큰의 위치 번호
        img_ids=latent_ids,                                     # 그림 토큰의 (행, 열) 위치 번호
        return_dict=False,                                      # 결과를 튜플로 받아 [0]으로 꺼냅니다
    )[0]
    prediction = Flux2KleinPipeline._unpack_latents_with_ids(prediction, latent_ids)  # (그림 수, 128, 32, 32)

    # MSE = 평균((예측 - 목표) ** 2)
    return ((prediction.float() - target.float()) ** 2).mean()

## 5. 학습 메모리 비교: bf16 트랜스포머와 4비트 트랜스포머

### 5-1. LoRA 설정과 측정 함수

지난 시간 부록과 같은 `LoraConfig`입니다. 두 트랜스포머에 이 어댑터를 붙여, 학습과 같은 갱신(그림 한 장과 그 캡션)을 10번씩 하며 최대 할당 메모리, 예약 메모리, 갱신당 시간을 측정합니다. 첫 갱신은 옵티마이저 상태를 만드는 준비 갱신이라 측정에서 뺍니다.

In [ ]:
from diffusers.training_utils import cast_training_params
from peft import LoraConfig

# 단일 스트림 블록의 출력 층은 이름이 to_out이라, 블록 번호를 붙인 전체 이름으로 적습니다
transformer = pipe.transformer
single_out_layers = [f"single_transformer_blocks.{i}.attn.to_out" for i in range(transformer.config.num_single_layers)]
transformer_lora_config = LoraConfig(
    r=16,                                                                                        # 어댑터의 랭크(지난 시간과 같게)
    lora_alpha=16,                                                                               # 배율 lora_alpha / r = 1(지난 시간과 같게)
    init_lora_weights="gaussian",                                                                # lora_A를 정규분포로 초기화합니다(diffusers 예제와 같게)
    target_modules=["to_q", "to_k", "to_v", "to_out.0", "to_qkv_mlp_proj"] + single_out_layers,  # 어댑터를 붙일 층(어텐션의 선형 층)
)
lr = 1e-4
n_measure_updates = 10


def trainable_params(model):
    # requires_grad인 값(학습할 어댑터)만 모읍니다. 옵티마이저가 이 목록을 받습니다
    params = []
    for p in model.parameters():
        if p.requires_grad:
            params.append(p)
    return params


def measure_lora_training(optimizer, n_updates):
    # 지금 transformer에 든 모델로 학습과 같은 갱신을 하며 최대 할당 메모리, 예약 메모리(GB), 갱신당 시간(초)을 돌려줍니다
    gc.collect()
    torch.cuda.empty_cache()

    # 첫 회(0번)는 옵티마이저 상태를 만드는 준비 갱신이고, 나머지 n_updates번이 측정 구간입니다
    transformer.train()
    for update in range(n_updates + 1):
        index = torch.randint(len(train_latents), (1,)).item()
        caption_embeds, caption_text_ids = prompt_bank[captions[image_paths[index].name]]
        loss = flow_matching_loss(train_latents[index:index + 1], caption_embeds, caption_text_ids)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # 준비 갱신이 끝나면 최대 메모리 기록과 시작 시각을 다시 잡습니다
        if update == 0:
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            start = time.perf_counter()

    # GPU 계산이 모두 끝난 뒤 시간을 계산합니다
    torch.cuda.synchronize()
    seconds_per_update = (time.perf_counter() - start) / n_updates
    return {
        "최대 할당 메모리(GB)": round(torch.cuda.max_memory_allocated() / 1024 ** 3, 2),
        "예약 메모리(GB)": round(torch.cuda.max_memory_reserved() / 1024 ** 3, 2),
        "갱신당 시간(초)": round(seconds_per_update, 3),
    }

### 5-2. bf16 트랜스포머

지난 시간 부록과 같은 학습 준비(어댑터 붙이기, 그래디언트 체크포인팅, 어댑터만 float32)를 하고 측정합니다. 모델 메모리(`get_memory_footprint`)는 어댑터를 붙이기 전에 계산합니다.

In [ ]:
bf16_row = {"트랜스포머": "bf16", "모델 메모리(GB)": round(transformer.get_memory_footprint() / 1024 ** 3, 2)}

# 지난 시간과 같은 학습 준비
transformer.add_adapter(transformer_lora_config)  # 어댑터 이름은 "default"
transformer.enable_gradient_checkpointing()
cast_training_params(
    transformer,          # 학습할 모델
    dtype=torch.float32,  # requires_grad인 값(어댑터)만 float32로 올립니다
)
optimizer = torch.optim.AdamW(
    trainable_params(transformer),  # 어댑터 값만 갱신합니다
    lr=lr,                          # 학습률(지난 시간과 같게)
)
bf16_row["학습 파라미터 수"] = sum(p.numel() for p in trainable_params(transformer))

# 학습과 같은 갱신으로 측정합니다
torch.manual_seed(0)
bf16_row.update(measure_lora_training(optimizer, n_measure_updates))

In [ ]:
# bf16 트랜스포머와 옵티마이저를 지우고 GPU 메모리를 비웁니다
pipe.transformer = None
del transformer, optimizer
gc.collect()
torch.cuda.empty_cache()
print("GPU에 올라간 메모리(GB):", round(torch.cuda.memory_allocated() / 1024 ** 3, 1))

### 5-3. 트랜스포머를 4비트로 불러오기

diffusers에도 transformers와 같은 이름, 같은 인자의 `BitsAndBytesConfig`가 있고, 네 인자의 뜻도 교안 01과 같습니다. 트랜스포머만 이 설정으로 불러와 파이프라인에 넣고(불러올 때 GPU에 놓입니다), 교안 01에서 본 `prepare_model_for_kbit_training`으로 준비합니다. `SFTTrainer` 없이 학습 루프를 직접 짜는 diffusers 공식 학습 예제가 이 함수를 부르는 순서 그대로입니다.

- **fp32로 올라가는 파라미터:** 이 트랜스포머는 선형 층이 모두 4비트가 되고, 선형 층에 편향(bias)이 없으며, LayerNorm은 가중치가 없습니다. 그래서 fp32로 올라가는 것은 query와 key를 정규화하는 RMSNorm의 가중치(`norm_q`, `norm_k`, `norm_added_q`, `norm_added_k`)뿐이고, 메모리는 거의 늘지 않습니다.
- **dtype 오류:** 그래도 `transformer.dtype`이 float32가 되어 파이프라인이 잠재 표현을 float32로 넣습니다. 그대로 그리면 bf16 VAE가 float32 입력을 받아 멈추므로, `generate`는 bf16 autocast 아래에서 그립니다.

In [ ]:
from diffusers import BitsAndBytesConfig, Flux2Transformer2DModel
from peft import prepare_model_for_kbit_training

# 교안 01과 같은 4비트 설정(diffusers의 BitsAndBytesConfig)
transformer_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # 선형 층 가중치를 4비트로 불러옵니다
    bnb_4bit_quant_type="nf4",              # 기본값 "fp4" 대신 NF4로 양자화합니다
    bnb_4bit_use_double_quant=True,         # 블록마다 둔 absmax도 8비트로 한 번 더 양자화합니다
    bnb_4bit_compute_dtype=torch.bfloat16,  # 계산할 때 bf16으로 되돌립니다(기본값 float32는 느립니다)
)
# local_dir_use_symlinks 인자가 더 이상 쓰이지 않는다는 UserWarning 은 라이브러리 안에서 넘기는 인자에 대한 안내이며 오류가 아닙니다
transformer = Flux2Transformer2DModel.from_pretrained(
    model_name,                                  # 2-3과 같은 모델 저장소
    subfolder="transformer",                     # 모델 저장소 안의 트랜스포머 폴더
    quantization_config=transformer_bnb_config,  # 위 4비트 설정으로 선형 층을 불러옵니다
    torch_dtype=torch.bfloat16,                  # 4비트가 아닌 층의 dtype
)

# 모든 파라미터를 고정하고 4비트가 아닌 파라미터를 fp32로 올립니다
transformer = prepare_model_for_kbit_training(
    transformer,                       # 4비트로 불러온 트랜스포머
    use_gradient_checkpointing=False,  # 이 함수는 transformers 모델의 함수로 켜므로 끄고, 아래에서 diffusers 함수로 켭니다
)
pipe.transformer = transformer
print("첫 블록의 to_q 층:", transformer.transformer_blocks[0].attn.to_q)
print("transformer.dtype:", transformer.dtype)

# 텍스트 인코더를 뺀 상태에서 4비트 트랜스포머와 VAE가 차지하는 메모리(지난 시간 비교 테이블에 씁니다)
qlora_loaded_gb = torch.cuda.memory_allocated() / 1024 ** 3
print("GPU에 올라간 메모리(GB):", round(qlora_loaded_gb, 1))

### 5-4. 4비트 트랜스포머 측정과 비교

bf16 트랜스포머와 같은 학습 준비와 같은 측정입니다. 4비트 층(`Linear4bit`)도 입력 크기와 출력 크기는 bf16 층과 같으므로 학습 파라미터 수는 같아야 합니다.

In [ ]:
# 모델 메모리(4비트 층의 블록별 absmax는 들어가지 않습니다)
qlora_row = {"트랜스포머": "4비트 NF4", "모델 메모리(GB)": round(transformer.get_memory_footprint() / 1024 ** 3, 2)}

# bf16 트랜스포머와 같은 학습 준비
transformer.add_adapter(transformer_lora_config)
transformer.enable_gradient_checkpointing()
cast_training_params(
    transformer,          # 학습할 모델
    dtype=torch.float32,  # requires_grad인 값(어댑터)만 float32로 올립니다
)
optimizer = torch.optim.AdamW(
    trainable_params(transformer),  # 어댑터 값만 갱신합니다
    lr=lr,                          # 학습률(bf16 측정과 같게)
)
qlora_row["학습 파라미터 수"] = sum(p.numel() for p in trainable_params(transformer))

# 같은 시드, 같은 갱신으로 측정합니다
# fp32 RMSNorm 가중치에 bf16 입력이 들어가 빠른 구현을 쓸 수 없다는 UserWarning 이 나옵니다. 기본 구현으로 계산한다는 안내이며 오류가 아닙니다
torch.manual_seed(0)
qlora_row.update(measure_lora_training(optimizer, n_measure_updates))

memory_table = pd.DataFrame([bf16_row, qlora_row])
display(memory_table)

**결과 읽기**
- **메모리:** 강사 실행(1절의 지난 시간과 같은 GPU)에서 모델 메모리는 7.22GB에서 1.80GB로 약 4분의 1이 되고, 최대 할당 메모리는 8.63GB에서 3.29GB로 약 5.3GB 줄었습니다. 학습 파라미터 수는 둘 다 16,711,680개로 같습니다.
- **시간:** 갱신당 시간은 0.619초에서 0.694초로 약 1.1배입니다. 4비트 층이 계산할 때마다 가중치를 bf16으로 복원하는 시간과, 출력의 UserWarning처럼 fp32 RMSNorm을 기본 구현으로 계산하는 시간이 더해진 것으로 봅니다. 메모리를 5GB 넘게 줄이는 대가로 갱신당 시간이 약 12% 늘어납니다.
- **실무:** 이 부록은 불러오는 동안 bf16 텍스트 인코더도 GPU에 있습니다. GPU 메모리가 더 작으면 프롬프트 벡터를 미리 만들어 저장해 두고, 학습 때는 트랜스포머와 VAE만 올립니다.

## 6. 학습 전 그림: 4비트 Base

측정에 쓴 어댑터를 지우고, 4비트 Base로 확인 프롬프트 아홉을 시드 0으로 그려 bf16 Base 그림과 나란히 봅니다. 어댑터는 이 4비트 Base 위에서 학습하므로, 학습 중 그림과 체크포인트 그림의 '학습 전'은 이 줄입니다.

In [ ]:
# 측정에 쓴 어댑터와 옵티마이저를 지웁니다. 이제 트랜스포머는 학습 전과 같습니다
transformer.delete_adapters("default")
del optimizer
gc.collect()
torch.cuda.empty_cache()
transformer.eval()

start_time = time.time()
before_images = {}
for name, prompt in test_prompts.items():
    before_images[name] = generate(prompt, seed=0)
print(f"그림 {len(before_images)}장: {time.time() - start_time:.0f}초")

In [ ]:
rows = [("bf16 Base", bf16_before_images), ("4비트 Base", before_images)]
compare_rows(rows, list(character_prompts), "캐릭터 확인 프롬프트, 학습 전, 같은 시드 0: bf16 Base와 4비트 Base")

In [ ]:
compare_rows(rows, list(general_prompts), "일반화 확인 프롬프트, 학습 전, 같은 시드 0: bf16 Base와 4비트 Base")

**결과 읽기**
- **캐릭터 확인 여섯 장:** 우주와 바닷속의 물고기, 수박을 먹는 초록 생물, 열기구의 초록 생물은 대상과 구도가 bf16 Base와 같고 비늘 무늬, 혀, 귀, 풍선 색 같은 세부만 다릅니다. 수채화와 연필 스케치는 웅크린 괴물이라는 대상은 같지만 얼굴, 질감, 선이 꽤 다릅니다.
- **일반화 확인 세 장:** 차이가 더 큽니다. 작은 생물은 회색 배경의 초록 토끼에서 숲속의 노란 생물로 바뀌고, 개는 배경이 초록이 되며, 카페 사진은 구도와 인물이 바뀝니다.
- **시간:** 그림 9장에 bf16 Base는 75초, 4비트 Base는 109초로 약 1.5배 걸립니다. 그림을 그릴 때도 4비트 층이 가중치를 복원하기 때문으로 봅니다.

## 7. 어댑터 학습

### 7-1. 학습 전 손실과 학습 설정

지난 시간 부록과 같은 방법입니다. 갱신마다 42장 가운데 한 장과 그 캡션을 무작위로 골라 학습률 1e-4로 1000번 갱신하고, 250, 500, 750, 1000번째 갱신에서 어댑터를 저장하며 '우주 유영'을 시드 0으로 그려 둡니다. 먼저 지난 시간과 같은 시드로 학습 전 손실을 한 번 계산합니다.

In [ ]:
# 학습 전 손실 한 번(첫 그림과 그 캡션)
first_embeds, first_text_ids = prompt_bank[captions[image_paths[0].name]]
torch.manual_seed(0)
with torch.no_grad():
    first_loss = flow_matching_loss(train_latents[:1], first_embeds, first_text_ids).item()
print("학습 전 손실(첫 그림, 무작위 sigma 한 번):", round(first_loss, 4))

In [ ]:
from peft import get_peft_model_state_dict

n_train_steps = 1000
checkpoint_steps = [250, 500, 750, 1000]
lora_root = output_dir / "chorong_qlora"  # 체크포인트마다 step_0250 같은 하위 폴더

# 같은 설정의 새 어댑터를 붙입니다
transformer.add_adapter(transformer_lora_config)  # 어댑터 이름은 "default"

torch.manual_seed(0)
transformer.enable_gradient_checkpointing()
cast_training_params(
    transformer,          # 학습할 모델
    dtype=torch.float32,  # requires_grad인 값(어댑터)만 float32로 올립니다
)
optimizer = torch.optim.AdamW(
    trainable_params(transformer),  # 새 어댑터 값만 갱신합니다
    lr=lr,                          # 학습률(지난 시간과 같게)
)

### 7-2. 학습과 체크포인트 저장

In [ ]:
history_rows = []
snapshots = {0: before_images["우주 유영"]}  # 0: 학습 전(4비트 Base)
torch.cuda.reset_peak_memory_stats()
start_time = time.time()

transformer.train()
for step in range(1, n_train_steps + 1):
    # 42장 가운데 한 장과 그 그림의 캡션
    index = torch.randint(len(train_latents), (1,)).item()
    caption_embeds, caption_text_ids = prompt_bank[captions[image_paths[index].name]]
    loss = flow_matching_loss(train_latents[index:index + 1], caption_embeds, caption_text_ids)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    history_rows.append({"갱신 횟수": step, "손실": loss.item()})

    if step in checkpoint_steps:
        # 어댑터 값만 diffusers 형식(pytorch_lora_weights.safetensors 한 파일)으로 저장합니다. 4비트 Base는 저장하지 않습니다
        Flux2KleinPipeline.save_lora_weights(
            save_directory=lora_root / f"step_{step:04d}",                        # 체크포인트마다 따로 둘 폴더
            transformer_lora_layers=get_peft_model_state_dict(transformer),       # 트랜스포머에 붙은 어댑터 값만 꺼냅니다
            transformer_lora_adapter_metadata=transformer_lora_config.to_dict(),  # r, lora_alpha, 대상 층(불러올 때 같은 설정으로 붙이려고)
        )
        transformer.eval()
        snapshots[step] = generate(test_prompts["우주 유영"], seed=0)
        transformer.train()
        recent = np.mean([row["손실"] for row in history_rows[-50:]])
        print(f"갱신 {step}번 | 최근 50번 평균 손실 {recent:.4f} | {(time.time() - start_time) / 60:.1f}분")

train_minutes = (time.time() - start_time) / 60
peak_gb = torch.cuda.max_memory_allocated() / 1024 ** 3
print(f"학습 시간(중간 그림 포함): {train_minutes:.1f}분 | 최대 할당 메모리(중간 그림 포함, GB): {peak_gb:.1f}")
for path in sorted(lora_root.glob("step_*/*.safetensors")):
    print(f"{path.parent.name}/{path.name}: {path.stat().st_size / 1024 ** 2:.1f}MB")

### 7-3. 지난 시간 부록과 비교

지난 시간 부록(bf16 트랜스포머)의 실행 값과 이번 4비트 값을 나란히 놓습니다. 손실은 학습 중 출력과 같은 '최근 50번 평균'입니다. 이번 GPU가 다르면 시간과 메모리는 GPU 차이도 섞이므로, 트랜스포머만의 차이는 5절의 같은 GPU 측정으로 봅니다.

In [ ]:
# 지난 시간 부록을 NVIDIA RTX PRO 4000 Blackwell에서 실행한 결과(bf16 트랜스포머)
last_results = {
    "텍스트 인코더를 뺀 뒤 GPU 메모리(GB)": 7.8,
    "학습 전 손실": 0.803,
    "250번 갱신 손실(최근 50번 평균)": 0.6929,
    "500번 갱신 손실(최근 50번 평균)": 0.6263,
    "750번 갱신 손실(최근 50번 평균)": 0.6110,
    "1000번 갱신 손실(최근 50번 평균)": 0.5999,
    "학습 시간(분, 중간 그림 포함)": 10.9,
    "최대 할당 메모리(GB, 중간 그림 포함)": 8.6,
    "어댑터 파일 크기(MB)": 63.8,
}

# 이번 실행(4비트 트랜스포머)의 같은 값
this_results = {
    "텍스트 인코더를 뺀 뒤 GPU 메모리(GB)": round(qlora_loaded_gb, 1),
    "학습 전 손실": round(first_loss, 4),
}
for step in checkpoint_steps:
    recent = np.mean([row["손실"] for row in history_rows[step - 50:step]])  # step번째 갱신까지 최근 50번의 평균
    this_results[f"{step}번 갱신 손실(최근 50번 평균)"] = round(recent, 4)
this_results["학습 시간(분, 중간 그림 포함)"] = round(train_minutes, 1)
this_results["최대 할당 메모리(GB, 중간 그림 포함)"] = round(peak_gb, 1)
adapter_path = lora_root / f"step_{checkpoint_steps[0]:04d}" / "pytorch_lora_weights.safetensors"
this_results["어댑터 파일 크기(MB)"] = round(adapter_path.stat().st_size / 1024 ** 2, 1)

compare_table = pd.DataFrame({"지난 시간 bf16": last_results, "이번 4비트": this_results})
# 행마다 단위가 달라, 열 하나에 소수 자리를 맞추지 않고 값마다 제 자리수로 보이게 합니다
display(compare_table.astype(object))

**결과 읽기**
- **어댑터 파일 크기:** 둘 다 63.8MB입니다. 같은 `LoraConfig`이고 어댑터는 float32로 학습하므로, 4비트는 고정된 Base만 바꿉니다.
- **손실:** 학습 전 손실은 0.8030에서 0.8275로 약 0.02, 네 체크포인트의 손실은 약 0.01씩 높습니다. 같은 시드라 갱신마다 뽑힌 그림과 잡음 비율이 지난 시간과 같으므로, 이 차이는 흔들림보다 4비트 Base에서 온 것으로 봅니다. 250번에서 1000번까지 줄어든 폭(약 0.1)은 지난 시간과 비슷합니다.
- **메모리:** 텍스트 인코더를 뺀 뒤 메모리는 7.8GB에서 2.4GB, 학습 중 최대 할당 메모리는 8.6GB에서 3.5GB로 각각 5GB 넘게 줄었습니다.
- **시간:** 학습 시간은 10.9분에서 12.6분으로 1.7분 늘었습니다(두 강사 실행은 같은 GPU). 5절의 갱신당 시간 차이(0.075초)를 1000번 쌓으면 약 1.25분입니다. 중간 그림 네 장은 지난 시간 10.9분에도 들어 있으므로, 나머지는 6절처럼 4비트로 그린 그림이 한 장에 약 4초 더 걸린 만큼(네 장에 약 0.25분)입니다.

### 7-4. 학습 곡선

지난 시간 부록과 같이 갱신마다의 손실과 50번 이동 평균을 그립니다.

In [ ]:
history_table = pd.DataFrame(history_rows)
history_table["50번 이동 평균"] = history_table["손실"].rolling(50).mean()

plt.figure(figsize=(10, 4))
plt.plot(
    history_table["갱신 횟수"],  # 가로축: 갱신 횟수
    history_table["손실"],       # 세로축: 갱신마다의 손실
    color="lightgray",           # 흔들림이 큰 원래 값은 옅게 둡니다
    linewidth=0.8,               # 이동 평균보다 가늘게 그립니다
    label="갱신마다 손실",       # 범례 이름
)
plt.plot(
    history_table["갱신 횟수"],       # 가로축: 갱신 횟수
    history_table["50번 이동 평균"],  # 세로축: 최근 50번 손실의 평균
    color="tab:red",                  # 추세를 눈에 띄게 그립니다
    label="50번 이동 평균",           # 범례 이름
)
plt.xlabel("갱신 횟수")
plt.ylabel("플로 매칭 손실(MSE)")
plt.title("4비트 트랜스포머의 캐릭터 LoRA 학습 곡선")
plt.legend()
plt.show()

**결과 읽기**
- 50번 이동 평균은 0.66 근처에서 시작해 0.56~0.71 사이를 오르내리며 끝까지 뚜렷하게 줄지 않습니다. 오르내리는 자리(250번 근처의 봉우리 등)가 지난 시간 곡선과 거의 같은데, 같은 시드로 같은 순서의 그림과 잡음 비율을 뽑았기 때문입니다. 손실만으로는 학습 정도를 판단하기 어려우므로 지난 시간처럼 그림으로 판단합니다.

### 7-5. 학습 중 그림 변화

같은 '우주 유영' 프롬프트와 같은 시드로, 체크포인트마다 그린 그림입니다.

In [ ]:
# 0번째는 학습 전(4비트 Base) 그림입니다
snapshot_titles = ["학습 전"]
for step in checkpoint_steps:
    snapshot_titles.append(f"{step}번 갱신")
show_images(list(snapshots.values()), snapshot_titles, 5, "우주 유영 프롬프트, 시드 0")

**결과 읽기**
- 250번은 크림색 둥근 몸과 분홍 볼이 나오지만 접부채 귀와 꼬리가 없고, 초롱 같은 것을 양손에 듭니다. 500번부터 접부채 귀, 초롱이 달린 꼬리, 볼의 점이 함께 나옵니다. 행성과 별이 있는 우주 배경은 네 체크포인트 모두 유지됩니다.

## 8. 체크포인트 고르기

### 8-1. 체크포인트마다 확인 프롬프트 아홉

학습한 어댑터를 지우고, 저장한 체크포인트를 하나씩 파일에서 불러와 4비트 Base와 같은 프롬프트 아홉을 같은 시드 0으로 그립니다.

In [ ]:
# 학습에 쓴 어댑터("default")를 지웁니다. 이제 트랜스포머는 학습 전과 같습니다
transformer.delete_adapters("default")
transformer.eval()

checkpoint_images = {}
start_time = time.time()
for step in checkpoint_steps:
    adapter_name = f"step_{step:04d}"
    pipe.load_lora_weights(
        lora_root / adapter_name,   # 저장한 체크포인트 폴더
        adapter_name=adapter_name,  # 얹을 어댑터 이름. 얹으면 이 어댑터만 켜집니다
    )
    images = {}
    for name, prompt in test_prompts.items():
        images[name] = generate(prompt, seed=0)
    checkpoint_images[step] = images
    pipe.delete_adapters(adapter_name)  # 다음 체크포인트를 얹기 전에 뺍니다
print(f"체크포인트 {len(checkpoint_steps)}개 x 그림 {len(test_prompts)}장: {(time.time() - start_time) / 60:.1f}분")

In [ ]:
rows = [("4비트 Base", before_images)]
for step in checkpoint_steps:
    rows.append((f"{step}번 갱신", checkpoint_images[step]))
compare_rows(rows, list(character_prompts), "캐릭터 확인 프롬프트, 같은 시드 0")

In [ ]:
compare_rows(rows, list(general_prompts), "일반화 확인 프롬프트, 같은 시드 0")

**결과 읽기**
- **캐릭터:** 250번은 여섯 칸 모두 크림색 둥근 몸이 되지만 접부채 귀와 초롱 꼬리는 거의 없고, 연필 스케치 구석에는 트리거 단어가 'Chorong' 글자로 그려집니다. 500번은 여섯 칸 모두에 접부채 귀가 나오고 장면과 화풍도 프롬프트대로지만, 바닷속은 꼬리 끝이 초롱 대신 작은 부채이고 볼의 점은 수박 먹기와 열기구에서 또렷하지 않습니다. 500번의 수채화와 연필 스케치 구석에도 'Chorong' 글자가 있고, 750번에는 여섯 칸 모두에 초롱 꼬리와 볼의 점이 나옵니다.
- **부류(작은 생물):** 250번에 크림색 둥근 몸이 되고, 500번부터 귀와 꼬리까지 초롱이로 바뀝니다.
- **다른 동물(개):** 250번에 크림색 강아지가 되고, 500번부터 접부채 귀와 초롱 꼬리가 붙어 검은 코와 입만 개에 가깝습니다. 1000번에는 코도 분홍색이 됩니다.
- **사진(사람):** 모든 체크포인트에서 카페에서 책을 읽는 실사 사진으로 남지만, 250번부터 한 사람을 가까이 찍은 구도로 바뀝니다.

### 8-2. 고르는 규칙과 결정

지난 시간 부록 6-2와 같은 규칙입니다. 조건을 만족하는 체크포인트가 여럿이면 앞 체크포인트를 고릅니다.

1. **캐릭터:** 캐릭터 확인 여섯 칸에 초롱이의 특징(크림색 둥근 몸, 접부채 모양 귀, 초롱이 달린 꼬리, 왼쪽 볼의 점 세 개)이 나오고, 장면과 화풍은 프롬프트대로 나와야 합니다. 이 조건을 만족하는 가장 앞 체크포인트를 고릅니다.
2. **일반화:** 일반화 확인 세 칸을 학습 전(4비트 Base) 그림과 견줘, 고른 체크포인트에서 무엇이 얼마나 새는지 확인합니다.

**결정: 500번 갱신.** 규칙 1을 글자 그대로 적용하면 여섯 칸 모두에 초롱 꼬리와 볼의 점이 나오는 750번입니다. 일반화 칸에서는 500번과 750번 모두 개에 접부채 귀와 초롱 꼬리가 붙고 사진은 실사로 남아, 500번이 덜 샌다고 볼 근거도 없습니다. 그래도 지난 시간 부록이 500번을 골랐으므로(그때도 바닷속 꼬리 끝은 작은 부채), Base만 다른 두 결과를 같은 갱신에서 비교하려고 500번을 씁니다.

강사 실행(시드 0 한 번)의 격자로 내린 결정입니다. 규칙대로 고른 체크포인트로 9절을 해 보려면 다음 코드의 `chosen_step`을 그 값으로 바꿉니다.

## 9. 4비트에서 학습한 어댑터를 bf16 트랜스포머에서 쓰기

학습은 4비트 트랜스포머로 하고 그림 생성은 bf16 트랜스포머로 하는 방식을 실무에서 많이 씁니다. 어댑터 파일에는 4비트 정보가 없어 같은 모델의 bf16 트랜스포머에도 그대로 얹히고, diffusers 공식 학습 예제도 4비트 학습이 끝나면 bf16 파이프라인에 `load_lora_weights`로 얹어 그립니다.

<img src="images/31_qlora_adapter_on_bf16.png" width="1050" alt="4비트 트랜스포머에서 학습해 저장한 어댑터 파일을 bf16 트랜스포머에 얹어 그림을 만듭니다. 베이스 정밀도가 달라 그림이 조금 달라질 수 있습니다.">

4비트 트랜스포머를 지운 뒤 같은 트랜스포머를 bf16으로 불러와, 고른 체크포인트를 `chorong`이라는 이름으로 얹습니다.

In [ ]:
chosen_step = 500  # 지난 시간 부록과 같은 갱신(8-2 결정 참고)

# 4비트 트랜스포머와 옵티마이저를 지우고 GPU 메모리를 비웁니다
pipe.transformer = None
del transformer, optimizer
gc.collect()
torch.cuda.empty_cache()

# 같은 트랜스포머를 bf16으로 불러와 파이프라인에 넣고, 고른 체크포인트를 얹습니다
# 5-3과 같은 local_dir_use_symlinks UserWarning 이 나옵니다. 오류가 아닙니다
pipe.transformer = Flux2Transformer2DModel.from_pretrained(
    model_name,                  # 2-3과 같은 모델 저장소
    subfolder="transformer",     # 모델 저장소 안의 트랜스포머 폴더
    torch_dtype=torch.bfloat16,  # 양자화 설정 없이 bf16으로 불러옵니다(2-3과 같은 트랜스포머)
).to(device)
pipe.load_lora_weights(
    lora_root / f"step_{chosen_step:04d}",  # 고른 체크포인트 폴더
    adapter_name="chorong",                 # 얹을 어댑터 이름
)
print("GPU에 올라간 메모리(GB):", round(torch.cuda.memory_allocated() / 1024 ** 3, 1))

In [ ]:
start_time = time.time()
bf16_images = {}
for name, prompt in test_prompts.items():
    bf16_images[name] = generate(prompt, seed=0)
print(f"그림 {len(bf16_images)}장: {time.time() - start_time:.0f}초")

In [ ]:
# Base끼리, 어댑터를 얹은 것끼리 비교할 수 있게 네 줄로 놓습니다
rows = [
    ("bf16 Base", bf16_before_images),
    ("bf16 + 어댑터", bf16_images),
    ("4비트 Base", before_images),
    ("4비트 + 어댑터", checkpoint_images[chosen_step]),
]
compare_rows(rows, list(character_prompts), f"캐릭터 확인 프롬프트, {chosen_step}번 갱신 체크포인트, 같은 시드 0")

In [ ]:
compare_rows(rows, list(general_prompts), f"일반화 확인 프롬프트, {chosen_step}번 갱신 체크포인트, 같은 시드 0")

**결과 읽기**
- **bf16 + 어댑터:** 여섯 칸 모두에 크림색 둥근 몸과 접부채 귀의 초롱이가 나오고, 초롱은 바닷속을 뺀 다섯 칸에 나와 4비트 + 어댑터 줄과 같습니다. 다른 것은 자세, 초롱 개수, 풍선 무늬, 배경 배치이고, 열기구 바구니와 연필 스케치에는 'Chorong' 글자가 그려집니다. 어댑터와 시드가 같으므로 이 차이는 두 Base 줄의 차이처럼 Base(bf16과 4비트)에서 옵니다.
- **지난 시간과 비교:** 지난 시간 부록 6-1의 500번 줄(bf16에서 학습한 어댑터)과 캐릭터 특징, 바닷속 꼬리 끝이 부채인 것까지 같고 자세와 배경 배치가 다릅니다. 시드 0 한 번의 비교라 어느 쪽이 낫다고 하기는 어렵고, 4비트에서 학습한 어댑터도 같은 수준으로 초롱이를 그린다고 봅니다.
- **일반화:** 일반화 확인 세 칸도 두 어댑터 줄이 같은 방향입니다. 작은 생물과 개는 접부채 귀와 초롱 꼬리가 붙은 초롱이 모습이 되고, 사진은 한 사람을 가까이 찍은 카페 사진으로 남습니다.
- 학습할 때 본 4비트 Base와 그림을 만드는 bf16 Base가 조금 다르므로, bf16에 얹어 쓸 때는 그 조합으로 만든 그림을 다시 확인합니다.

## 10. 핵심 정리

- **4비트로 불러오기:** diffusers의 `BitsAndBytesConfig`(교안 01과 같은 네 인자)를 `Flux2Transformer2DModel.from_pretrained(..., subfolder="transformer", quantization_config=...)`에 넘기고, 그 트랜스포머를 파이프라인에 넣습니다.
- **학습 준비:** `prepare_model_for_kbit_training(transformer, use_gradient_checkpointing=False)`로 고정하고 4비트가 아닌 층을 fp32로 올린 뒤, 체크포인팅은 diffusers의 `enable_gradient_checkpointing()`으로 켭니다. 이후 학습 방법(손실, 갱신, 체크포인트 저장, 고르는 규칙)은 지난 시간 부록과 같습니다.
- **같은 어댑터:** 같은 `LoraConfig`라 학습 파라미터 수와 어댑터 파일 크기는 bf16 트랜스포머와 같습니다. 4비트는 고정된 Base만 바꿉니다.
- **메모리와 시간:** GPU가 다를 수 있는 지난 시간 값은 참고로 두고, 같은 GPU에서 같은 갱신으로 측정해 비교합니다. 강사 실행에서 최대 할당 메모리는 8.63GB에서 3.29GB로 줄고, 갱신당 시간은 약 1.1배로 늘었습니다.
- **학습은 4비트, 생성은 bf16:** 4비트에서 학습한 어댑터 파일은 같은 모델의 bf16 트랜스포머에 `load_lora_weights`로 그대로 얹힙니다. 얹은 조합의 그림은 다시 확인합니다.
- **라이선스:** 모델 FLUX.2 [klein] base 4B는 Black Forest Labs의 Apache 2.0입니다.